In [1]:
import re
import pandas as pd
import numpy as np
from critdd import Diagram

def clean_latex_value(val):
    """Strips LaTeX commands like \best{0.9}, \third{0.8}, etc. and returns a float."""
    # Remove any LaTeX command like \best{...} or \third{...} but keep the content inside { }
    val = re.sub(r'\\[a-zA-Z]+\{([^}]+)\}', r'\1', val)
    # Remove any remaining LaTeX commands like \textbf, \textit, \underline
    val = re.sub(r'\\[a-zA-Z]+', '', val)
    # Remove braces and spaces
    val = val.replace('{', '').replace('}', '').strip()
    # Handle '-' or empty strings as NaN
    if val == '-' or val == '' or 'multirow' in val:
        return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan

def parse_latex_to_df(latex_text):
    # The columns as defined in your table header
    methods = ["DTW", "GOW", "POW", "OPW", "TAOT", "TCOT", "ASW", "OTW", "TMW-1", "TMW-2"]
    
    acc_rows = []
    map_rows = []
    
    current_dataset = None
    
    # Split by LaTeX line breaks
    lines = latex_text.split(r'\\')
    
    for line in lines:
        # Clean up the line
        line = line.strip()
        if '&' not in line:
            continue
            
        parts = [p.strip() for p in line.split('&')]
        
        # Check if this line starts a new dataset (multirow)
        # Structure: \multirow{2}{*}{NAME} & METRIC & V1 & V2 ...
        if 'multirow' in parts[0]:
            # Extract dataset name from inside the last set of curly braces
            match = re.findall(r'\{([^}]+)\}', parts[0])
            if match:
                current_dataset = match[-1]
            metric = parts[1].upper()
            values = parts[2:]
        else:
            # Structure: & METRIC & V1 & V2 ... (or sometimes METRIC is parts[0])
            # If parts[0] is empty or just whitespace, it's the second row of the multirow
            if parts[0] == '' or parts[0].isspace():
                metric = parts[1].upper()
                values = parts[2:]
            else:
                # Fallback for weirdly formatted rows
                metric = parts[0].upper()
                values = parts[1:]

        # Convert values to numbers
        numeric_values = [clean_latex_value(v) for v in values]
        
        # Ensure we only take the number of values matching our methods list
        numeric_values = numeric_values[:len(methods)]
        
        row_data = [current_dataset] + numeric_values
        
        if "ACC" in metric:
            acc_rows.append(row_data)
        elif "MAP" in metric:
            map_rows.append(row_data)

    # Create DataFrames
    columns = ["Dataset"] + methods
    df_acc = pd.DataFrame(acc_rows, columns=columns).set_index("Dataset")
    df_map = pd.DataFrame(map_rows, columns=columns).set_index("Dataset")
    
    return df_acc, df_map

def run_analysis(df, name):
    print(f"\n--- Analysis for {name} ---")
    
    # CRITDD cannot handle NaNs. 
    # Option 1: Drop datasets with any missing values (like Weizmann/MSR/SAD)
    clean_df = df.dropna()
    
    if clean_df.empty:
        print(f"Skipping {name}: No datasets remain after dropping rows with missing values.")
        return

    print(f"Processing {len(clean_df)} datasets...")
    
    # Initialize Diagram
    diagram = Diagram(
        clean_df.to_numpy(),
        treatment_names = clean_df.columns,
        maximize_outcome = True
    )
    
    # Export
    output_file = f"cdd_{name.lower()}.tex"
    diagram.to_file(
        output_file,
        alpha = 0.05,
        adjustment = "holm",
        reverse_x = True
    )
    print(f"Successfully saved to {output_file}")
    print("Average Ranks:")
    print(diagram.average_ranks)

# --- YOUR DATA ---
latex_input = r"""
\begin{sidewaystable} 
	\centering
	\footnotesize
	\caption{Classification of the $k$-NN with different distance measures. The optimal $k$ for each method was chosen based on the best balanced performance of Accuracy and mAP. The top three highest scores for each dataset are presented in bold, italic red and underlined blue, respectively.} \label{tab:clustering_results}
	\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|c|}
		\hline
		\multicolumn{2}{|c|}{Datasets} & DTW & GOW & POW & OPW & TAOT & TCOT & ASW & OTW & TMW-1 & TMW-2 \\
		\hline
		\hline
		\multirow{2}{*}{UMD} 
		& ACC & 0.861 & 0.542 & 0.590 & 0.826 & 0.792 & 0.861 & \third{0.889} & 0.715 & \second{0.958} & \best{0.993} \\
		& MAP & \third{0.867} & 0.452 & 0.492 & 0.742 & 0.702 & 0.793 & 0.835 & 0.612 & \second{0.933} & \best{0.989} \\
		\hline
		\multirow{2}{*}{BME} 
		& ACC & 0.893 & 0.493 & 0.687 & \third{0.913} & 0.740 & 0.647 & 0.800 & 0.740 & \second{0.960} & \best{0.980} \\
		& MAP & \third{0.917} & 0.436 & 0.611 & 0.868 & 0.653 & 0.596 & 0.847 & 0.700 & \second{0.937} & \best{0.967} \\
		\hline
		\multirow{2}{*}{CT} 
		& ACC & \best{0.974} & 0.714 & 0.417 & 0.714 & 0.889 & 0.878 & 0.921 & \third{0.965} & \second{0.971} & 0.895 \\
		& MAP & \third{0.951} & 0.657 & 0.484 & 0.666 & 0.806 & 0.794 & 0.895 & \second{0.961} & \best{0.972} & 0.869 \\
		\hline
		\multirow{2}{*}{SS} 
		& ACC & 0.893 & 0.587 & \second{0.993} & \third{0.987} & 0.927 & 0.933 & 0.920 & 0.887 & \best{1.000} & 0.980 \\
		& MAP & 0.951 & 0.524 & \second{0.999} & 0.996 & 0.965 & 0.946 & 0.950 & 0.831 & \best{1.000} & \third{0.997} \\
		\hline
		\multirow{2}{*}{DLD} 
		& ACC & 0.455 & 0.506 & 0.494 & 0.403 & 0.519 & \second{0.571} & 0.481 & 0.429 & \second{0.571} & \best{0.610} \\
		& MAP & 0.446 & 0.413 & 0.423 & 0.287 & 0.491 & 0.441 & \second{0.524} & 0.413 & \best{0.527} & \third{0.517} \\
		\hline
		\multirow{2}{*}{DLW} 
		& ACC & 0.960 & 0.937 & \third{0.968} & 0.952 & \third{0.968} & 0.960 & \third{0.968} & 0.960 & \best{0.976} & \best{0.976} \\
		& MAP & \second{0.967} & 0.878 & \third{0.958} & \third{0.958} & 0.942 & 0.926 & \best{0.968} & 0.950 & 0.955 & 0.955 \\
		\hline
		\multirow{2}{*}{SGWZ} 
		& ACC & \best{0.880} & 0.280 & 0.700 & \third{0.860} & 0.620 & 0.840 & 0.620 & 0.740 & 0.840 & \best{0.880} \\
		& MAP & \third{0.854} & 0.193 & 0.582 & \second{0.856} & 0.598 & 0.837 & 0.669 & 0.651 & 0.751 & \best{0.893} \\
		\hline
		\multirow{2}{*}{PGWZ} 
		& ACC & \best{0.780} & 0.260 & 0.480 & 0.720 & 0.700 & 0.660 & 0.540 & 0.580 & \third{0.740} & \best{0.780} \\
		& MAP & \second{0.794} & 0.231 & 0.484 & 0.682 & 0.567 & \third{0.693} & 0.436 & 0.476 & 0.610 & \best{0.825} \\
		\hline
		\multirow{2}{*}{CBF} 
		& ACC & \second{0.997} & 0.854 & 0.837 & \second{0.997} & 0.932 & 0.722 & 0.924 & 0.891 & \best{0.999} & 0.992 \\
		& MAP & \best{1.000} & 0.788 & 0.768 & \third{0.995} & 0.897 & 0.654 & 0.964 & 0.885 & \second{0.998} & 0.987 \\
		\hline
		\multirow{2}{*}{L7} 
		& ACC & \second{0.767} & 0.178 & 0.658 & 0.726 & 0.699 & 0.603 & 0.712 & 0.644 & \second{0.767} & \best{0.808} \\
		& MAP & \best{0.882} & 0.164 & 0.635 & \third{0.750} & 0.699 & 0.575 & 0.700 & 0.602 & 0.713 & \second{0.788} \\
		\hline
		\multirow{2}{*}{TS1} 
		& ACC & 0.772 & 0.706 & 0.772 & \third{0.776} & 0.768 & \second{0.816} & 0.754 & 0.671 & 0.759 & \best{0.833} \\
		& MAP & 0.712 & 0.654 & 0.715 & 0.720 & 0.713 & \third{0.759} & \best{0.814} & 0.612 & 0.706 & \second{0.786} \\
		\hline
		\multirow{2}{*}{TS2} 
		& ACC & 0.846 & 0.869 & 0.769 & \best{0.938} & 0.908 & 0.800 & \third{0.915} & 0.800 & \third{0.915} & \second{0.931} \\
		& MAP & 0.755 & 0.680 & 0.686 & \third{0.880} & 0.817 & 0.688 & 0.876 & 0.628 & \second{0.889} & \best{0.891} \\
		\hline
		\multirow{2}{*}{GPZ1} 
		& ACC & \second{0.715} & 0.657 & 0.576 & 0.634 & \third{0.709} & 0.634 & 0.616 & 0.558 & \best{0.733} & 0.674 \\
		& MAP & \third{0.737} & 0.665 & 0.582 & 0.478 & 0.721 & 0.717 & 0.473 & 0.393 & \best{0.815} & \second{0.765} \\
		\hline
		\multirow{2}{*}{GPZ2} 
		& ACC & 0.728 & 0.741 & 0.665 & 0.677 & \best{0.804} & \second{0.785} & 0.620 & 0.601 & \second{0.785} & 0.703 \\
		& MAP & 0.710 & \third{0.773} & 0.528 & 0.714 & \best{0.837} & \second{0.821} & 0.706 & 0.436 & 0.769 & 0.740 \\
		\hline
		\multirow{2}{*}{FST} 
		& ACC & 0.763 & 0.700 & \third{0.767} & \third{0.767} & 0.766 & 0.766 & 0.765 & \best{0.772} & \second{0.771} & \third{0.767} \\
		& MAP & \second{0.765} & 0.672 & 0.756 & 0.757 & \third{0.762} & \best{0.779} & \third{0.762} & 0.761 & 0.710 & 0.706 \\
		\hline
		\multirow{2}{*}{FRT} 
		& ACC & 0.899 & 0.534 & 0.764 & \second{0.935} & 0.774 & \third{0.914} & 0.886 & 0.851 & \best{0.944} & 0.902 \\
		& MAP & 0.859 & 0.584 & 0.840 & \second{0.907} & 0.778 & 0.879 & 0.843 & 0.800 & \best{0.919} & \third{0.904} \\
		\hline
		\multirow{2}{*}{MP} 
		& ACC & 0.879 & 0.755 & 0.086 & 0.727 & 0.876 & 0.879 & 0.809 & \best{0.898} & \second{0.896} & \third{0.884} \\
		& MAP & \best{0.890} & 0.786 & 0.102 & 0.749 & \third{0.880} & \second{0.882} & 0.829 & 0.813 & 0.810 & 0.790 \\
		\hline
		\multirow{2}{*}{GPMVF} 
		& ACC & 0.984 & \third{0.994} & 0.525 & 0.975 & \second{0.997} & \best{1.000} & \best{1.000} & 0.978 & \second{0.997} & \best{1.000} \\
		& MAP & \third{0.976} & \second{0.995} & 0.577 & 0.963 & \best{1.000} & \best{1.000} & \best{1.000} & 0.967 & \best{1.000} & \best{1.000} \\
		\hline
		\multirow{2}{*}{GPAS} 
		& ACC & \second{0.994} & \best{0.997} & 0.538 & 0.981 & \third{0.987} & \third{0.987} & \third{0.987} & 0.959 & \best{0.997} & \second{0.994} \\
		& MAP & \best{1.000} & \best{1.000} & 0.535 & 0.972 & 0.981 & 0.981 & 0.989 & 0.940 & \second{0.995} & \third{0.991} \\
		\hline
		\multirow{2}{*}{Trace} 
		& ACC & \best{1.000} & 0.310 & 0.740 & 0.970 & 0.700 & \third{0.980} & \third{0.980} & 0.760 & \second{0.990} & \second{0.990} \\
		& MAP & \best{1.000} & 0.508 & 0.781 & 0.947 & 0.628 & 0.963 & \third{0.965} & 0.663 & \second{0.981} & \second{0.981} \\
		\hline
        \hline
		\multirow{2}{*}{Weizmann} 
		& ACC & 0.900 & \best{0.967} & \best{0.967} & 0.900 & 0.933 & \best{0.967} & 0.400 & - & \best{0.967} & \best{0.967} \\
		& MAP & 0.947 & 0.962 & \second{0.983} & \best{1.000} & 0.962 & \third{0.967} & 0.565 & - & 0.962 & 0.962 \\
		\hline
		% \multirow{2}{*}{MSR} 
		% & ACC & \best{0.714} & 0.650 & 0.657 & \second{0.680} & 0.377 & 0.640 & 0.310 & - & 0.640 & \third{0.670} \\
		% & MAP & \best{0.728} & 0.622 & \third{0.650} & \second{0.661} & 0.341 & 0.620 & 0.253 & - & 0.625 & 0.636 \\
		% \hline
        \multirow{2}{*}{BasicMotions} 
		& ACC & 0.975 & 0.750 & 0.775 & \best{1.000} & \best{1.000} & \best{1.000} & 0.250 & - & \best{1.000} & \best{1.000} \\
		& MAP & 0.959 & 0.722 & 0.998 & \best{1.000} & \best{1.000} & \best{1.000} & 0.311 & - & \best{1.000} & \best{1.000} \\
        \hline
		\multirow{2}{*}{CharTraj} 
		& ACC & \second{0.988} & 0.112 & 0.943 & \best{0.989} & 0.974 & 0.972 & 0.976 & - & 0.982 & \third{0.985} \\
		& MAP & 0.976 & 0.162 & 0.956 & \third{0.991} & 0.990 & 0.987 & \best{0.995} & - & \second{0.992} & 0.987 \\
        \hline  
		\multirow{2}{*}{SAD} 
		& ACC & \best{0.973} & 0.802 & 0.908 & 0.938 & 0.237 & 0.943 & 0.241 & - & \third{0.954} & \second{0.955} \\
		& MAP & \best{0.994} & 0.880 & 0.957 & \third{0.976} & 0.233 & 0.969 & 0.214 & - & 0.975 & \second{0.976} \\
        \hline
	\end{tabular} 
\end{sidewaystable}

"""

# Note: Paste your full string into latex_input above.
# I'm calling the parser here:
df_acc, df_map = parse_latex_to_df(latex_input)

# Run for both metrics
run_analysis(df_acc, "ACC")
run_analysis(df_map, "MAP")


--- Analysis for ACC ---
Processing 20 datasets...
Successfully saved to cdd_acc.tex
Average Ranks:
[4.45  8.05  7.675 5.425 5.65  5.525 5.975 7.1   2.475 2.675]

--- Analysis for MAP ---
Processing 20 datasets...
Successfully saved to cdd_map.tex
Average Ranks:
[3.475 8.4   7.575 5.425 5.65  5.425 4.775 7.675 3.35  3.25 ]


c:\Users\tungv\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
c:\Users\tungv\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
